# CD-CFL — Committee-Driven Byzantine Defense for Federated Learning

Run CDCFL-I / CDCFL-II / CMFL experiments on Google Colab with GPU.

**Methods:**
| Name | Pipeline |
|------|----------|
| FedAvg | Plain averaging (no defense) |
| CMFL (Strategy I) | L2 scoring → top-α selection → pBFT |
| CMFL-II (Strategy II) | L2 scoring → bottom-α selection → pBFT |
| CDCFL-I | Robust Agg (all gradients) → Consensus Check → PoW/pBFT finalization |
| CDCFL-II | PoW pre-filter → Outlier Filter → Robust Agg → pBFT |

**Experiments:**
| ID | Name | Description |
|----|------|-------------|
| E1 | Convergence | No attack — all methods |
| E2 | Robustness | All methods under 3 Byzantine attacks at ε=10% |
| E3 | Design Choice Ablation | E3-A: CDCFL-II layer ablation, E3-B: CDCFL-I vs CDCFL-II strategy comparison |
| E4 | PoW Effectiveness | PoW rejection analysis (from E2, no new runs) |
| E5 | Stress Test | All methods at ε=10%–50% |
| E6 | Convergence Speed | No attack — convergence comparison |
| E7 | Overhead | Computational overhead (from E2, no new runs) |

**Setup:** `Runtime → Change runtime type → GPU (T4 or better)`

## 1. Clone Repository & Install Dependencies

In [ ]:
# Clone the repository
!git clone https://github.com/RahulMadhukar/Research-Work.git /content/Research-Work
%cd /content/Research-Work/federated_Defense

In [ ]:
# Install dependencies (most are pre-installed on Colab)
!pip install -q torch torchvision numpy scipy matplotlib seaborn

In [ ]:
# Verify GPU is available
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime → Change runtime type → GPU")

## 2. Configuration

Adjust these settings before running experiments.

**Datasets:** `femnist`, `shakespeare`, `sentiment140` (or `None` for all)

**Attacks:** `gradient_scaling`, `same_value`, `back_gradient` (or `None` for all)

**Experiments:** `E1`, `E2`, `E3`, `E4`, `E5`, `E6`, `E7` (comma-separated, or `None` for all)

In [ ]:
# ========== CONFIGURATION ==========

# Dataset filter (None = run all 3 datasets)
DATASET = 'femnist'           # 'femnist', 'shakespeare', 'sentiment140', or None

# Attack filter (None = run all 3 attacks)
ATTACK  = 'gradient_scaling'  # 'gradient_scaling', 'same_value', 'back_gradient', or None

# Experiment filter (None = run all E1-E7)
EXPERIMENT = 'E2'             # 'E1', 'E2', 'E3', 'E2,E3', 'E5', etc., or None

# ===================================
print(f"Dataset:     {DATASET or 'ALL (FEMNIST, Shakespeare, Sentiment140)'}")
print(f"Attack:      {ATTACK or 'ALL (gradient_scaling, same_value, back_gradient)'}")
print(f"Experiments: {EXPERIMENT or 'ALL (E1-E7)'}")
print()
print("Methods that will run:")
print("  FedAvg, CMFL, CMFL-II, CDCFL-I, CDCFL-II")
print("  (+ CDCFL-I pBFT and layer ablation variants in E3)")

## 3. (Optional) Tune Hyperparameters

Default values (paper-exact):
- FEMNIST: lr=0.01, batch_size=32, rounds=100
- Shakespeare: lr=0.001, batch_size=32, rounds=100
- Sentiment140: lr=0.005, batch_size=32, rounds=100
- Clients: 100, Active per round: 10%, Data fraction: 50%

In [ ]:
# Show current hyperparameters
!grep -n 'DATASET_LR\|DATASET_BATCH_SIZE\|DATASET_ROUNDS' run_impact_analysis.py | head -5
print()
!grep -n 'ROUNDS\|FRACTION\|NUM_CLIENTS\|EPS_DEFAULT\|ALPHA\|OMEGA' lightweight_test.py | head -10

In [ ]:
# Uncomment to change hyperparameters programmatically:

# import lightweight_test, run_impact_analysis
#
# # Change rounds
# lightweight_test.ROUNDS = {'FEMNIST': 50, 'Shakespeare': 50, 'Sentiment140': 50}
#
# # Change data fraction (0.25 = 25% of data, faster)
# lightweight_test.FRACTION = 0.25
#
# # Change number of clients
# lightweight_test.NUM_CLIENTS = 50
#
# # Change learning rates
# run_impact_analysis.DATASET_LR['FEMNIST'] = 0.001

## 4. Run Experiments

In [ ]:
# Build and run the command
cmd = 'python lightweight_test.py'
if DATASET:
    cmd += f' -d {DATASET}'
if ATTACK:
    cmd += f' -a {ATTACK}'
if EXPERIMENT:
    cmd += f' -e {EXPERIMENT}'

print(f"Running: {cmd}")
print("Plots and tables will be auto-generated in Output/cdcfl_lightweight/{timestamp}/plots/\n")
!{cmd}

## 5. View Results

Plots (PNG) and tables (CSV) are auto-generated in `Output/cdcfl_lightweight/{timestamp}/plots/`.
The cells below let you also view them inline in Colab.

In [ ]:
# List output directories and generated plots
import os, glob

output_base = 'Output/cdcfl_lightweight'
if os.path.exists(output_base):
    runs = sorted(os.listdir(output_base))
    print(f"Found {len(runs)} run(s) in {output_base}/")
    for run_dir in runs[-3:]:  # show last 3
        run_path = os.path.join(output_base, run_dir)
        if not os.path.isdir(run_path):
            continue
        # Top-level files
        top_files = [f for f in os.listdir(run_path) if os.path.isfile(os.path.join(run_path, f))]
        # Plots subdir
        plots_path = os.path.join(run_path, 'plots')
        plot_files = os.listdir(plots_path) if os.path.isdir(plots_path) else []
        print(f"\n  {run_dir}/  ({len(top_files)} files, {len(plot_files)} plots)")
        for f in top_files:
            fsize = os.path.getsize(os.path.join(run_path, f))
            print(f"    {f}  ({fsize/1024:.1f} KB)")
        if plot_files:
            print(f"    plots/")
            for f in sorted(plot_files):
                fsize = os.path.getsize(os.path.join(plots_path, f))
                print(f"      {f}  ({fsize/1024:.1f} KB)")
else:
    print(f"No output yet at {output_base}/ — run experiments first (Section 4)")

In [ ]:
# Load and display results JSON
import json

output_base = 'Output/cdcfl_lightweight'
json_files = sorted(glob.glob(f"{output_base}/**/results.json", recursive=True))

if json_files:
    latest = json_files[-1]
    print(f"Loading: {latest}\n")
    with open(latest) as f:
        data = json.load(f)

    # Print summary table
    print(f"{'Key':<60s} | {'Accuracy':>10s} | {'ASR':>10s}")
    print("-" * 88)
    for key, val in data.items():
        if val is None:
            continue
        ds, method, attack, eps = key.split('|')
        acc = val.get('final_accuracy', 0)
        asr = val.get('asr', 0)
        label = f"{ds} | {method:<15s} | {attack:<18s} | eps={eps}"
        print(f"  {label:<58s} | {acc*100:>8.2f}% | {asr*100:>8.2f}%")
else:
    print("No results.json found yet — run experiments first (Section 4)")

## 6. Display Plots Inline

Auto-generated PNG plots are in the `plots/` subfolder. This cell displays them inline.
It also draws accuracy curves from the JSON data for any result groups.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import json, glob, os

output_base = 'Output/cdcfl_lightweight'

# --- Part 1: Show auto-generated PNG plots ---
png_files = sorted(glob.glob(f"{output_base}/**/plots/*.png", recursive=True))
if png_files:
    print(f"Displaying {len(png_files)} auto-generated plots:\n")
    for pf in png_files:
        img = mpimg.imread(pf)
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.imshow(img)
        ax.set_title(os.path.basename(pf), fontsize=10)
        ax.axis('off')
        plt.tight_layout()
        plt.show()
else:
    print("No auto-generated plots found.")

# --- Part 2: Draw accuracy curves from JSON ---
json_files = sorted(glob.glob(f"{output_base}/**/results.json", recursive=True))
if json_files:
    with open(json_files[-1]) as f:
        data = json.load(f)

    groups = {}
    for key, val in data.items():
        if val is None or not val.get('acc_history'):
            continue
        ds, method, attack, eps = key.split('|')
        gkey = (ds, attack, eps)
        if gkey not in groups:
            groups[gkey] = {}
        groups[gkey][method] = val['acc_history']

    for (ds, attack, eps), methods in sorted(groups.items()):
        fig, ax = plt.subplots(figsize=(10, 5))
        for method, acc_hist in sorted(methods.items()):
            ax.plot(range(1, len(acc_hist)+1),
                    [a*100 for a in acc_hist],
                    label=method.upper(), linewidth=1.5)
        atk_label = attack if attack != 'none' else 'no attack'
        ax.set_title(f"{ds} — {atk_label} (eps={eps})")
        ax.set_xlabel("Round")
        ax.set_ylabel("Test Accuracy (%)")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
else:
    print("No results.json found yet — run experiments first (Section 4)")

## 7. Save Results to Google Drive

In [ ]:
# Mount Google Drive and copy results
from google.colab import drive
drive.mount('/content/drive')

import shutil
src = '/content/Research-Work/federated_Defense/Output'
dst = '/content/drive/MyDrive/CDCFL_Results'

if os.path.exists(src):
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"Results saved to Google Drive: {dst}/")
else:
    print("No Output/ directory found — run experiments first")